In [ ]:
#import libraries
import os
import librosa
import numpy as np
import torch

In [ ]:
#extracting mfccs
target_sr=16000
n_mfc=20
max_frames=128
"""Loads an audio file, extracts MFCCs, and pads/truncates to MAX_FRAMES."""
def extract_fixed_mfcc(file_path):
    try:
        signal ,sr=librosa.load(file_path,sr=target_sr)
        mfcc = librosa.feature.mfcc(y=signal,sr=sr,n_mfcc=n_mfc,norm='orhto',lifter=22)
        #22 is considered the ideal lifter value for speech recognition tasks.
        num_rows,num_cols=mfcc.shape
        #slicing and padding the audio clip for matrix consistency
        if num_cols>max_frames :
            mfcc_fixed=[:,,:max_frames]
        else:
            pad_width=max_frames-num_cols
            mfcc_fixed=np.pad(mfcc,((0,0),(0,pad_width)),mode='constant')
            #0 rows to the top and bottom 0 columns to the left and  columns to the right
        return mfcc_fixed
    except Exception as e:
        print(f"Error processing {file_path}:{e}")
        

In [ ]:
#the tess dataset only consists of female voices so we must use both tess and ravdess dataset

import random
from collections import defaultdict
tess_dir='/tests'
ravdess_dir='.\ravdess'
emotion_map={
    "happy":0,
    "sad":1,
    "angry":2,
    "neutral":3
}
#the ravdess and tess datasets should give same labels for each emotions(0,1,2,3)
ravdess_emotion_map={
    "01":3,#neutral
    "03":0,#happy
    "04":1,#sad
    "05":2#angry
}

tess_files = defaultdict(list)
ravdess_files = defaultdict(list)

# 1. Collect TESS paths
for root, dirs, files in os.walk(tess_dir):
    for filename in files:
        if filename.endswith(".wav"):
            filepath = os.path.join(root, filename)
            for emotion, emotion_id in emotion_map.items():
                if emotion in filename.lower() or emotion in filepath.lower():
                    tess_files[emotion_id].append(filepath)
                    break

# 2. Collect RAVDESS paths
for root, dirs, files in os.walk(ravdess_dir):
    for filename in files:
        if filename.endswith(".wav"):
            parts = filename.split('-')
            if len(parts) >= 3:
                emotion_code = parts[2]
                if emotion_code in ravdess_emotion_map:
                    emotion_id = ravdess_emotion_map[emotion_code]
                    filepath = os.path.join(root, filename)
                    ravdess_files[emotion_id].append(filepath)

In [ ]:
#the tess+ravdess datset is quite large so we need to take only some of the samples
max_files_per_emotion = 96 # 96 is optimal because neutral in ravdess has only 96 files 

sampled_paths_with_labels = []

# Seed the random generator for consistent samples across runs
random.seed(42)

for keys,emotion_id in emotion_map.values():
    # Sample from TESS
    tess_pool = tess_files[emotion_id]
    sample_size_tess = min(max_files_per_emotion, len(tess_pool))
    sampled_tess = random.sample(tess_pool, sample_size_tess)
    for path in sampled_tess:
        sampled_paths_with_labels.append((path, emotion_id))
        
    # Sample from RAVDESS
    ravdess_pool = ravdess_files[emotion_id]
    sample_size_ravdess = min(max_files_per_emotion, len(ravdess_pool))
    sampled_ravdess = random.sample(ravdess_pool, sample_size_ravdess)
    for path in sampled_ravdess:
        sampled_paths_with_labels.append((path, emotion_id))

# Shuffle the final collection so TESS and RAVDESS files are intermixed
random.shuffle(sampled_paths_with_labels)

print(f"Total files selected for processing: {len(sampled_paths_with_labels)}")

In [ ]:
#extract mfcc features from the datset
x_features=[]
y_labels=[]


for pairs in sampled_paths_with_labels:
    filepath=pairs[0]
    emotion_id=pairs[1]
    found_emotion=None
    mfcc_fea=extract_fixed_mfcc(filepath)
    if mfcc_fea is not None:
        x_features.append(mfcc_fea)
        y_labels.append(emotion_id)

x_data=np.array(x_features,dtype= np.float32)
y_data=np.array(y_labels,dtype=torch.long)#torch.long=int64
#int64 is optimal as PyTorch's library is hardcoded for int64 and nn.crossentopy loss works best for int64


print(f"Final shape of the Features array:{x_data.shape}")
print(f"Final shape of the Labels array:{y_data.shape}")
        

In [ ]:
#make a custom dataset
import torch 
from torch.utils.data import Dataset,random_split,DataLoader


class Finalcustomdataset(Dataset):
    def __init__(self,features,labels):
        self.features=torch.tensor(features,dtype=np.float32)
        self.labels=torch.tensor(labels,dtype=torch.long)

    def len(self):
        return len(self.labels)

    def get_item(self,idx):
        x=self.features[idx] # Shape: (20, 128)
        y=self.labels[idx] 
        x=x.unsqueeze(0)   # Shape becomes: (1, 20, 128) -> Ready for 2D CNN
        return x,y


#instantiating the dataset
full_dataset=Finalcustomdataset(x.data,y.data)
total_size=len(full_dataset)
train_size=(int)(0.7*total_size)
test_size=(int)(0.15*total_size)
valid_size=(int)(0.15*total_size)
#chnage the device if running on kaggle
generator = torch.Generator().manual_seed(42) # Keeps splits identical across runs#torch.Generator() creates private randomness generator
train_dataset, val_dataset,test_dataset = random_split(full_dataset, [train_size, valid_size,test_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

#PRINT A CONFUSION MATRIX


In [ ]:
#the model
import torch.nn as nn
import torch.nn.functional as f
class emotionCNN(nn.module):
    def __init__(self,num_classes):
        super().__init__()

        self.layer1=nn.conv2d(in_channels=1,out_channels=16,kernel_size=3,stride=1,padding=1,dilation=(1,2))#dilation required for second dimension only for increasing receptive field
        self.maxpool1=nn.MaxPool2d(kernel_size=2,stride=2,ceil_mode=True)#ceil_mode=true for not skipping out edge values in the mfcc matrix

        self.layer2=nn.conv2d(in_channels=16,out_channels=32,kernel_size=3,stride=1,padding=1,dilation=(1,4))#dilation increased to compensate for the down sampling
        self.maxpool2=nn.MaxPool2d(kernel_size=2,stride=2,ceil_mode=True)

        self.adaptive_pool=nn.AdaptiveAvgPool2d((4,4))#makes the spatial feautres of each feature map consistent

        self.fc=nn.Linear(32*4*4,num_classes)
        #expects a 2d matrix of size (batchsize,inputfeatures)

        def forward(self,x):
            x=self.maxpool1(f.relu(self.conv1(x)))
            x=self.maxpool2(f.relu(self.conv2(x)))

            x=self.adaptive_pool(x)
            x==torch.flatten(x,1) #start_dim=1 skip batch size and flatten others reuired for nn.Linear
            
            x=self.fc(x)

            return x



    





In [ ]:
# Initialize device, model, loss, and optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = emotionCNN(num_classes=4).to(device) 
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 15

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_train_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * inputs.size(0)
        
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    
    # --- VALIDATION PHASE ---
    model.eval()
    running_val_loss = 0.0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)
            
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    
    # --- MONITOR OVERFITTING ---
    print(f"Epoch [{epoch+1}/{num_epochs}] -> Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    if epoch_val_loss > epoch_train_loss * 1.5:
        print("   ⚠️ Warning: Validation loss is significantly higher than train loss. Overfitting detected.")